# Synthetic stand-in, stronger run on a Colab T4

Runtime > Change runtime type > **T4 GPU**, then run every cell from top to bottom (about 55 minutes).

This run uses ImageNet initialisation, up to 48 patches per image, 8 epochs and **4 training seeds** (the `SYNTHETIC_T4` preset in `src/config.py`). It replaces `results/`, `checkpoints/` and `figures/`, and every number in the write-up updates from `results/numbers.tex`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Get the code: upload spoofbuster-lite-repro.zip when asked (or set REPO_URL to clone your repo)
REPO_URL = ''
import os, subprocess
os.chdir('/content')
if REPO_URL:
    subprocess.run(['git', 'clone', REPO_URL], check=True)
else:
    from google.colab import files
    up = files.upload()
    subprocess.run(['unzip', '-q', '-o', list(up)[0]], check=True)
%cd /content/spoofbuster-lite-repro

In [ ]:
# 2. Switch to the T4 preset (the only change to the code)
p = 'src/config.py'
s = open(p).read()
old = 'SYNTHETIC = SYNTHETIC_CPU'
assert old in s, 'preset line not found'
open(p, 'w').write(s.replace(old, 'SYNTHETIC = SYNTHETIC_T4', 1))
!grep -n '^SYNTHETIC = ' src/config.py

In [ ]:
# 3. Pin the versions THIS runtime uses (the results will come from it)
!python scripts/pin_env.py

In [ ]:
# 4. Everything: synthetic data, minutiae, 4 models, evaluation, timing, tables, figures
!bash run_all.sh --synthetic --retrain 2>&1 | grep -v -E 'TensorSpec|WARNING|I0000|E0000|W0000|^ +[0-9]+:'

In [ ]:
# 5. Check the results
!cat results/headline.json results/train_env.json
!for t in main ablation seeds timing material; do echo "== $t"; sed -n '/midrule/,/bottomrule/p' results/table_$t.tex; done

In [ ]:
# 6. Download the updated files: send results_t4.zip to Claude, then copy its contents into your repo
!zip -qr /content/results_t4.zip results checkpoints figures environment.yml src/config.py
from google.colab import files
files.download('/content/results_t4.zip')